# Vérification du téléchargement des zones réglementées / protégées

Ce notebook vérifie que `download.zones_protegees.download_zones_protegees` interroge correctement les couches de zones réglementées/protégées (source PatriNat/OFB, distinctes de BD TOPO : ZNIEFF 1, ZNIEFF 2, arrêtés de protection de biotope/habitats naturels, réserves biologiques, biens du patrimoine mondial UNESCO, périmètres de sites fragiles) sur l'emprise d'une entité du CSV, écrit chaque couche non vide en GeoPackage dans `data/raw/vector/zones_protegees/`, et affiche le résultat sur une carte.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities
from download.limites_administratives import fetch_commune_boundary, resolve_bbox
from download.zones_protegees import download_zones_protegees

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Téléchargement des zones réglementées/protégées sur la bbox de l'entité

In [ ]:
bbox = resolve_bbox(entity)
written = download_zones_protegees(bbox)
written

## 3. Relecture des GeoPackage écrits

In [ ]:
import geopandas as gpd

layers = {name: gpd.read_file(path) for name, path in written.items()}
for name, gdf in layers.items():
    print(name, "->", len(gdf), "entites")

## 4. Affichage sur une carte `leafmap`

Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium` (HTML statique), plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

styles = {
    "znieff1": {"color": "green", "fillColor": "green", "fillOpacity": 0.3},
    "znieff2": {"color": "darkgreen", "fillColor": "darkgreen", "fillOpacity": 0.2},
    "aire_protection_habitats_naturels": {"color": "olive", "fillColor": "olive", "fillOpacity": 0.3},
    "reserve_biologique": {"color": "darkred", "fillColor": "darkred", "fillOpacity": 0.3},
    "patrimoine_mondial_unesco": {"color": "gold", "fillColor": "gold", "fillOpacity": 0.3},
    "perimetre_site_fragile": {"color": "purple", "fillColor": "purple", "fillOpacity": 0.3},
}

boundary_wgs84 = fetch_commune_boundary(entity.code_insee).to_crs(epsg=4326)

m = leafmap.Map()

for name, gdf in layers.items():
    if gdf.empty:
        continue
    m.add_gdf(
        gdf.to_crs(epsg=4326),
        layer_name=name,
        style=styles.get(name),
        info_mode="on_click",
    )

m.add_gdf(
    boundary_wgs84,
    layer_name="Limite communale",
    style={"color": "black", "fillOpacity": 0},
)
m.zoom_to_gdf(boundary_wgs84)
m